In [1]:
from rich.console import Console
from dotenv import load_dotenv
from openai import OpenAI
import json
load_dotenv(override=True)

True

In [4]:
def show(text):
    try:
        Console().print(text)
    except Exception:
        print(text)

In [5]:
import os
openai = OpenAI(
    api_key=os.getenv("DEEPSEEK_API_KEY"),
    base_url=os.getenv("DEEPSEEK_BASE_URL"),
)

In [14]:
checklist = []
completed = []

In [15]:
def get_checklist_report() -> str:
    result = ""
    for index, item in enumerate(checklist):
        if completed[index]:
            result += f"Checklist #{index + 1}: [green][strike]{item}[/strike][/green]\n"
        else:
            result += f"Checklist #{index + 1}: {item}\n"
    show(result)
    return result


In [9]:
get_checklist_report()

''

In [25]:
def create_checklist(descriptions: list[str]) -> str:
    checklist.extend(descriptions)
    completed.extend([False] * len(descriptions))
    return get_checklist_report()


In [11]:
def mark_complete(index: int, completion_notes: str) -> str:
    if 1 <= index < len(checklist):
        completed[index-1] = True
    else:
        return "No checklist at this index"
    Console().print(completion_notes)
    return get_checklist_report()

In [16]:
checklist, completed = [], []

create_checklist(["Buy groceries", "Finish week 1", "Eat banana"])

Checklist #1: Buy groceries
Checklist #2: Finish week 1
Checklist #3: Eat banana

'Checklist #1: Buy groceries\nChecklist #2: Finish week 1\nChecklist #3: Eat banana\n'

In [17]:
mark_complete(1, "bought")

bought

Checklist #1: Buy groceries
Checklist #2: Finish week 1
Checklist #3: Eat banana

'Checklist #1: [green][strike]Buy groceries[/strike][/green]\nChecklist #2: Finish week 1\nChecklist #3: Eat banana\n'

In [18]:
create_checklist_json = {
    "name": "create_checklist",
    "description": "Add new checklist from a list of descriptions and return the full list",
    "parameters": {
        "type": "object",
        "properties": {
            "descriptions": {
                'type': 'array',
                'items': {'type': 'string'},
                'title': 'Descriptions of checklist items'
                }
            },
        "required": ["descriptions"],
        "additionalProperties": False
    }
}

In [19]:
mark_complete_json = {
    "name": "mark_complete",
    "description": "Mark complete the checklist item at the given position (starting from 1) and return the full list",
    "parameters": {
        'properties': {
            'index': {
                'description': 'The 1-based index of the checklist item to mark as complete',
                'title': 'Index',
                'type': 'integer'
                },
            'completion_notes': {
                'description': 'Notes about how you completed the checklist item in rich console markup',
                'title': 'Completion Notes',
                'type': 'string'
                }
            },
        'required': ['index', 'completion_notes'],
        'type': 'object',
        'additionalProperties': False
    }
}

In [20]:
tools = [{"type": "function", "function": create_checklist_json},
        {"type": "function", "function": mark_complete_json}]

In [21]:
def handle_tool_calls(tool_calls):
    results = []
    for tool_call in tool_calls:
        tool_name = tool_call.function.name
        arguments = json.loads(tool_call.function.arguments)
        tool = globals().get(tool_name)
        result = tool(**arguments) if tool else {}
        results.append({"role": "tool","content": json.dumps(result),"tool_call_id": tool_call.id})
    return results

In [22]:
model = os.getenv("DEEPSEEK_MODEL")
def loop(messages):
    response = openai.chat.completions.create(model=model, messages=messages, tools=tools)
    while response.choices[0].finish_reason == "tool_calls":
        message = response.choices[0].message
        tool_calls = message.tool_calls
        results = handle_tool_calls(tool_calls)
        messages.append(message)
        messages.extend(results)
        response = openai.chat.completions.create(model=model, messages=messages, tools=tools)
    show(response.choices[0].message.content)

In [23]:
system_message = """
You are given a problem to solve, by using your checklist tools to plan a list of steps, then carrying out each step in turn.
Now create a plan, set the checklist, carry out the steps, and reply with the solution.
If any quantity isn't provided in the question, then include a step to come up with a reasonable estimate.
Provide your solution in Rich console markup without code blocks.
Do not ask the user questions or clarification; respond only with the answer after using your tools.
"""
user_message = """"
A train leaves Boston at 2:00 pm traveling 60 mph.
Another train leaves New York at 3:00 pm traveling 80 mph toward Boston.
When do they meet?
"""
messages = [{"role": "system", "content": system_message}, {"role": "user", "content": user_message}]

In [26]:
checklist, completed = [], []
loop(messages)

Checklist #1: Identify unknowns and estimate the Boston–New York distance
Checklist #2: Compute the Boston train's head-start distance by 3:00 pm
Checklist #3: Compute the remaining gap and closing speed at 3:00 pm
Checklist #4: Calculate time after 3:00 pm until meeting and convert to a clock time
Checklist #5: Assemble final answer with assumptions and formula

The distance is not given, so I estimate the Boston–New York distance as 215 miles, a common approximate road 
distance.

Checklist #1: Identify unknowns and estimate the Boston–New York distance
Checklist #2: Compute the Boston train's head-start distance by 3:00 pm
Checklist #3: Compute the remaining gap and closing speed at 3:00 pm
Checklist #4: Calculate time after 3:00 pm until meeting and convert to a clock time
Checklist #5: Assemble final answer with assumptions and formula

From 2:00 pm to 3:00 pm, the Boston train travels 60 mph × 1 h = 60 miles.

Checklist #1: Identify unknowns and estimate the Boston–New York distance
Checklist #2: Compute the Boston train's head-start distance by 3:00 pm
Checklist #3: Compute the remaining gap and closing speed at 3:00 pm
Checklist #4: Calculate time after 3:00 pm until meeting and convert to a clock time
Checklist #5: Assemble final answer with assumptions and formula

...

Checklist #1: Identify unknowns and estimate the Boston–New York distance
Checklist #2: Compute the Boston train's head-start distance by 3:00 pm
Checklist #3: Compute the remaining gap and closing speed at 3:00 pm
Checklist #4: Calculate time after 3:00 pm until meeting and convert to a clock time
Checklist #5: Assemble final answer with assumptions and formula

Checklist #1: Identify unknowns and estimate the Boston–New York distance
Checklist #2: Compute the Boston train's head-start distance by 3:00 pm
Checklist #3: Compute the remaining gap and closing speed at 3:00 pm
Checklist #4: Calculate time after 3:00 pm until meeting and convert to a clock time
Checklist #5: Assemble final answer with assumptions and formula
Checklist #6: Assemble final answer with assumptions and formula

Final answer: about 4:06 pm, assuming Boston–New York is about 215 miles.

Checklist #1: Identify unknowns and estimate the Boston–New York distance
Checklist #2: Compute the Boston train's head-start distance by 3:00 pm
Checklist #3: Compute the remaining gap and closing speed at 3:00 pm
Checklist #4: Calculate time after 3:00 pm until meeting and convert to a clock time
Checklist #5: Assemble final answer with assumptions and formula
Checklist #6: Assemble final answer with assumptions and formula

They meet at approximately 4:06 pm — more precisely, about 4:06:26 pm — assuming the Boston–New York distance is 
roughly 215 miles.

At 3:00 pm, the Boston train has already traveled:

60 mph × 1 h = 60 miles

So the remaining distance between the trains is:

215 − 60 = 155 miles

Their closing speed is:

60 + 80 = 140 mph

Time until they meet after 3:00 pm:

155 ÷ 140 ≈ 1.107 hours ≈ 1 h 6 min 26 s

Thus:

3:00 pm + 1 h 6 min 26 s ≈ 4:06:26 pm

General formula if the Boston–New York distance is D miles:

Time after 3:00 pm = (D − 60) / 140 hours